# 라벨링 스키마와 실행 조건

출력 스키마를 검증하고 dry-run manifest를 확인합니다. **API를 호출하지 않습니다.**

## 주 라벨은 가격 산정 가능성입니다

PROJECT_DIRECTION §5.1의 핵심 질문에서 나옵니다 —
*"이 요구사항을 그대로 수용한 상태에서 믿을 만한 금액과 수행 범위를 산정할 수 있는가?"*

| 라벨 | 기준 |
|---|---|
| `통상수용` | 별도 추가 원가가 붙지 않는다. 기본 수행팀 공수에 포함 |
| `견적반영` | 추가 원가가 붙지만 원문 정보만으로 계산할 수 있다 |
| `계약·질의검토` | 원가를 계산할 수 없다. 범위·책임·기준이 닫혀 있지 않다 |

**부담이 크다는 것과 계산이 불가능하다는 것은 다른 사실입니다.** 부담이 커도 계산되면 `견적반영`입니다.

## 보조 축 4개

같은 `계약·질의검토`라도 이유가 다르면 실무 조치가 다릅니다. 그 이유를 분리해 기록합니다.

| 필드 | 값 | 역할 |
|---|---|---|
| `blockers` | 범위·책임 / 검수·성능기준 / 기술실현성 / 라이선스·공급 / 공급자종속 | 입찰 전 확인이 필요한 조건. 복수 가능, 빈 배열이면 없음 |
| `cost_basis` | 없음 / 고급·전문인력 / 장비·인프라 / 라이선스 / 외부인증 / 외주·전문기관 / 복합 | 추가 원가의 계산 근거 |
| `domain_dependency` | 높음 / 보통 / 낮음 | 발주기관 업무 지식 없이 수행이 막히는 정도 |
| `build_difficulty` | 높음 / 보통 / 낮음 | 도메인 지식을 모두 제공받는다고 가정한 순수 구축 난이도 |

`blockers`는 **비용이 든다는 뜻이 아니라 제안·입찰 전에 반드시 확인해야 안전하게 수용할 수 있는 조건**을 뜻합니다.

## 버전 이력

스키마와 프롬프트는 별개로 버전이 매겨집니다. v5 프롬프트는 스키마 v4를 그대로 쓰면서
표준 문구 보정만 추가한 것이라 스키마 버전이 오르지 않았습니다.
상세 근거는 `docs/history/decisions-02.md` 결정 16~22를 보세요.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from scripts.labeling.label_schema import (
    BLOCKER_TYPES,
    PRIMARY_ACTIONS,
    REASONING_MAX_LENGTH,
    SCHEMA_VERSION,
    LabelResult,
    derive_primary_action,
)
from scripts.labeling.claude_client import PROMPT_VERSION
from scripts.labeling.validate_label_schema import validate_label_output
from scripts.labeling.run_claude_labeling import build_parser, make_manifest

print(f'스키마 {SCHEMA_VERSION} / 프롬프트 {PROMPT_VERSION}')
print(f'주 라벨: {PRIMARY_ACTIONS}')
print(f'blocker 범주: {BLOCKER_TYPES}')
print(f'reasoning 최대 길이: {REASONING_MAX_LENGTH}자')
print(f'\n필수 필드: {LabelResult.model_json_schema()["required"]}')

In [ ]:
# 정상 출력 예시. LLM이 실제로 돌려주는 형태와 같습니다.
valid = {
    'requirement_uid': 'demo-SFR-001',
    'primary_action': '견적반영',
    'blockers': [],                       # 빈 배열 = 입찰 전 확인이 필요한 조건 없음
    'cost_basis': '고급·전문인력',          # 원가는 붙는다
    'domain_dependency': '보통',
    'build_difficulty': '높음',
    'reasoning': '다단계 추론 로직 구현에 고급 데이터·AI 개발 공수가 필요하나, 범위와 완료 조건이 명확해 산정 가능하다.',
}
ok, errors = validate_label_output(valid)
print(f'정상 예시 검증: {ok}')

# 구 스키마(v2)는 거부됩니다. extra="forbid"이므로 없어진 필드가 있으면 걸리고,
# 새로 생긴 필수 필드가 빠져도 걸립니다. 서로 다른 버전의 결과가
# 한 데이터셋에 섞이는 것을 막기 위한 장치입니다.
legacy_v2 = {
    'requirement_uid': 'demo-SFR-001',
    'primary_action': '통상수용',
    'confidence': '높음',                  # v2에 있었으나 결정 17에서 제거된 필드
    'reasoning': '조회 화면 제공만 요구하므로 일반적인 SI 개발 범위에 해당한다.',
}
ok_legacy, errors_legacy = validate_label_output(legacy_v2)
print(f'\n구 스키마(v2) 검증: {ok_legacy}')
for e in errors_legacy:
    print(f'  - {e}')

In [ ]:
# 주 라벨은 보조 축에서 규칙으로 도출할 수도 있습니다 (결정 21).
#   blocker 있음                    -> 계약·질의검토
#   blocker 없음 + 원가 있음        -> 견적반영
#   blocker 없음 + 원가 없음        -> 통상수용
#
# LLM이 직접 출력한 primary_action과 이 규칙의 결과를 비교하면
# "직접 3분류"와 "위험 요인 분해 후 규칙 도출"을 추가 API 호출 없이 비교할 수 있습니다.
cases = [
    ('원가도 blocker도 없음', dict(valid, primary_action='통상수용', blockers=[], cost_basis='없음')),
    ('원가만 있음', dict(valid, primary_action='견적반영', blockers=[], cost_basis='장비·인프라')),
    ('blocker 있음', dict(valid, primary_action='계약·질의검토', blockers=['라이선스·공급'], cost_basis='복합')),
    ('난이도만 높음', dict(valid, primary_action='통상수용', blockers=[], cost_basis='없음',
                        build_difficulty='높음', domain_dependency='높음')),
]
print(f"{'상황':<22}{'LLM 직접':<14}{'규칙 도출':<14}일치")
for name, data in cases:
    label = LabelResult.model_validate(data)
    derived = derive_primary_action(label)
    print(f'{name:<22}{label.primary_action:<14}{derived:<14}{label.primary_action == derived}')

# 마지막 사례가 중요합니다. 난이도가 높다는 사실만으로 주 라벨을 올리지 않습니다.
# 높은 난이도가 고급·전문 인력 투입으로 이어질 때만 견적반영이 됩니다.

In [ ]:
# dry-run manifest. --execute 없이 실행 조건만 출력합니다.
# 이 manifest가 재현의 기준이며, 조건이 다르면 같은 output-dir에 이어쓰기가 거부됩니다.
import json

args = build_parser().parse_args(['--limit', '1'])
manifest = make_manifest([valid], args)
print(json.dumps(manifest, ensure_ascii=False, indent=2))

print('\n실행 명령 (기본은 dry-run이며 --execute를 붙여야 실제 호출)')
print('  python -m scripts.labeling.run_claude_labeling --limit 3')
print('  python -m scripts.labeling.run_claude_labeling --limit 3 --execute')
print('\n앵커를 주입하는 few-shot 전략')
print('  python -m scripts.labeling.run_claude_labeling \\')
print('    --strategy fewshot-stratified \\')
print('    --anchor-pool data/anchors/anchor_pool_v2.jsonl --limit 3')